# Problem Set 8: Legal Drinking Age and Mortality

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due Wednesday 11/18 (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

A regression discontinuity analysis of what happens to mortality the moment alcohol becomes legal.

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up

This problem set is based on Carpenter and Dobkin (2009), "The Effect of Alcohol Consumption on Mortality: Regression Discontinuity Evidence from the Minimum Drinking Age," *AEJ: Applied* 1(1): 164–182 — one of the cleanest and most famous applications of **regression discontinuity** in economics.

The rule could not be simpler: in the US you may legally purchase and consume alcohol only once you turn 21. Age is the **running variable**, turning 21 is the **cutoff**, and legal drinking access is the **treatment**, which switches sharply from 0 to 1 exactly at the cutoff. Carpenter and Dobkin ask: since legal access causes a large jump in drinking (recent drinking rises about 21% right at 21), does it also cause a detectable jump in **mortality**?

> **About the data.** The underlying mortality microdata are restricted-access vital statistics records. We instead use the exact dataset the authors constructed and published: mortality rates (deaths per 100,000 person-years) aggregated into fine, approximately monthly **age-cell bins**, by cause of death, covering ages 19–23. This is real, publicly archived data (it also appears in Angrist and Pischke's *Mastering 'Metrics*). Because it is already aggregated, you run regressions directly on the 50 age-cell observations.

| Variable | Description |
|---|---|
| `agecell` | Age (years, two decimals) at the centre of the bin |
| `all` | All-cause mortality rate (per 100,000 person-years) |
| `internal` | Internal causes (cancer, heart disease, …) |
| `external_cause` | External causes (accidents, poisonings, violence, suicide) |
| `alcohol`, `homicide`, `suicide`, `mva`, `drugs`, `externalother` | Specific causes |

Each outcome also has a `*fitted` counterpart containing the **authors' own fitted values** — useful later as a check on your code.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

## Part 1: Visualizing the discontinuity

**Question 1.** Load and describe the data. How many age-cell bins are there, what is the spacing between bins, and do the data span ages 19 to 23?

In [ ]:
mlda = pd.read_csv("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/mlda_mortality.csv")

print("Bins:", len(mlda))
print(f"Age range: {mlda.agecell.min():.3f} to {mlda.agecell.max():.3f}")
print(f"Spacing between bins: {mlda.agecell.diff().median():.4f} years "
      f"({mlda.agecell.diff().median()*12:.1f} months)")
mlda[["agecell", "all", "internal", "external_cause", "mva", "alcohol"]].head()

**Answer:** *[your answer here]*

**Question 2.** Plot all-cause mortality against age, with a vertical line at 21.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(mlda.agecell, mlda["all"])
plt.axvline(21, color="gray", ls="--")
plt.xlabel("Age"); plt.ylabel("All-cause mortality rate (per 100,000)")
plt.title("Mortality around the minimum legal drinking age")
plt.show()

**Question 3. Just from this picture, does it look like there is a discontinuity at age 21?** Describe what you see in a sentence or two.

**Answer:** *[your answer here]*

## Part 2: The sharp RD regression

We estimate the discontinuity with a **local polynomial regression**: fit a smooth polynomial in age separately on each side of the cutoff, and read the gap between the fitted lines at 21 as the RD estimate.

**Question 4.** Construct the RD variables. **Explain in your own words what each does, and why we need the interaction terms specifically** (without them, what would we be assuming about the slope of mortality with respect to age on either side?).

In [ ]:
mlda["post"] = (mlda.agecell >= 21).astype(int)
mlda["linear"] = mlda.agecell - 21          # age centred at the cutoff
mlda["square"] = mlda.linear ** 2
mlda["linear_post"] = mlda.linear * mlda.post
mlda["square_post"] = mlda.square * mlda.post

mlda[["agecell", "post", "linear", "square", "linear_post", "square_post"]].iloc[22:28]

**Answer:** *[your answer here]*

**Question 5.** Run the RD regression — a quadratic polynomial allowed to differ on each side of the cutoff, the same specification as the authors' own replication code. **What is the coefficient on `post`, and how do you interpret it?** Express it as a percentage of the pre-cutoff baseline and compare to the paper's headline finding of a **9 percent** increase.

In [ ]:
rd_all = smf.ols("Q('all') ~ post + linear + square + linear_post + square_post",
                 mlda).fit(cov_type="HC1")
print(rd_all.summary().tables[1])

baseline = mlda.loc[mlda.post == 0, "all"].mean()
jump = rd_all.params["post"]
print(f"\nPre-cutoff baseline mortality: {baseline:.2f} per 100,000")
print(f"Estimated jump at 21:         {jump:.2f} per 100,000")
print(f"As a percentage of baseline:  {100*jump/baseline:.1f}%   (paper: ~9%)")

**Answer:** *[your answer here]*

**Question 6.** Overlay your fitted values on the scatter, alongside the authors' own `allfitted` values, as a check that your regression matches the original analysis. **Do they line up?**

In [ ]:
mlda["allhat"] = rd_all.predict(mlda)

plt.figure(figsize=(9, 5))
plt.scatter(mlda.agecell, mlda["all"], label="Raw data")
for side in (0, 1):
    s = mlda[mlda.post == side]
    plt.plot(s.agecell, s.allhat, color="C1",
             label="Your fitted values" if side == 0 else None)
    plt.plot(s.agecell, s.allfitted, color="C2", ls="--",
             label="Authors' fitted values" if side == 0 else None)
plt.axvline(21, color="gray", ls="--")
plt.xlabel("Age"); plt.ylabel("All-cause mortality rate (per 100,000)")
plt.legend(); plt.show()

print("Max |your fit - authors' fit|:", round((mlda.allhat - mlda.allfitted).abs().max(), 3))

**Answer:** *[your answer here]*

## Part 3: Decomposing the effect by cause of death

**Question 7.** Repeat the regression for each cause of death and build a summary table reporting the jump, the baseline mean, and the percentage change.

In [ ]:
causes = ["all", "internal", "external_cause", "alcohol", "homicide",
          "suicide", "mva", "drugs", "externalother"]

rows = []
for c in causes:
    r = smf.ols(f"Q('{c}') ~ post + linear + square + linear_post + square_post",
                mlda).fit(cov_type="HC1")
    base = mlda.loc[mlda.post == 0, c].mean()
    rows.append({"cause": c, "jump at 21": r.params["post"], "t-stat": r.tvalues["post"],
                 "baseline": base, "% of baseline": 100 * r.params["post"] / base})

rd_table = pd.DataFrame(rows).set_index("cause")
rd_table.round(2)

**Question 8. Which causes drive the overall jump, and which do not?** Compare the `internal` estimate to `external_cause` in particular. Why would we *not* expect legal alcohol access to affect internal-cause deaths (cancer, heart disease) in the short run, and why does that make `internal` a useful **falsification/placebo test** for the whole design? **What would you conclude about the design's credibility if internal-cause mortality had jumped just as much?**

**Answer:** *[your answer here]*

**Question 9.** Make a bar chart comparing the estimated jumps in percentage-of-baseline terms across causes. **Which single cause contributes most to the overall effect?**

In [ ]:
plot_causes = rd_table.drop(index="all").sort_values("% of baseline")

plt.figure(figsize=(9, 5))
colors = ["C3" if c == "internal" else "C0" for c in plot_causes.index]
plt.barh(plot_causes.index, plot_causes["% of baseline"], color=colors)
plt.axvline(0, color="black", lw=1)
plt.xlabel("Estimated jump at 21 (% of pre-cutoff baseline)")
plt.title("Which causes of death drive the mortality jump? (red = placebo)")
plt.tight_layout(); plt.show()

# in absolute terms, which cause contributes the most deaths?
plot_causes[["jump at 21"]].sort_values("jump at 21", ascending=False).round(2)

**Answer:** *[your answer here]*

## Part 4: How sensitive is the estimate to specification choices?

**Question 10. Bandwidth.** Restrict to ages 20–22 and re-estimate. **How does the estimate change? What is the general trade-off in choosing a narrower versus wider bandwidth** (think bias versus precision)?

In [ ]:
narrow = mlda[(mlda.agecell >= 20) & (mlda.agecell <= 22)]
rd_narrow = smf.ols("Q('all') ~ post + linear + square + linear_post + square_post",
                    narrow).fit(cov_type="HC1")

print(f"Full window (19-23): {rd_all.params['post']:.2f}  "
      f"(se {rd_all.bse['post']:.2f}, n = {int(rd_all.nobs)})")
print(f"Narrow (20-22):      {rd_narrow.params['post']:.2f}  "
      f"(se {rd_narrow.bse['post']:.2f}, n = {int(rd_narrow.nobs)})")

**Answer:** *[your answer here]*

**Question 11. Polynomial order.** Re-estimate with only a **linear** polynomial on each side. **Does the estimate change much? When might linear and quadratic fits give meaningfully different answers?**

In [ ]:
rd_linear = smf.ols("Q('all') ~ post + linear + linear_post", mlda).fit(cov_type="HC1")

print(f"Quadratic: {rd_all.params['post']:.2f}  (se {rd_all.bse['post']:.2f})")
print(f"Linear:    {rd_linear.params['post']:.2f}  (se {rd_linear.bse['post']:.2f})")

**Answer:** *[your answer here]*

## Part 5: Interpreting the design

**Question 12. Sharp versus fuzzy RD.** Strictly speaking, is legal drinking *access* sharp or fuzzy at 21? Now consider actual *drinking behaviour* — is the jump in the probability that a given person drinks sharp or fuzzy? **What would you need (that this dataset lacks) to set up a fuzzy RD / IV design scaling the effect of legal access up to the effect of actually drinking?**

**Answer:** *[your answer here]*

**Question 13. RD validity checks.** We normally check that pre-determined characteristics and the number of observations are smooth around the cutoff, as evidence it is not being manipulated. **Is there anything a person could do to manipulate which side of the age-21 cutoff they fall on?** Why does this make you more or less confident here than in, say, an RD based on a test-score cutoff students might strategically influence?

**Answer:** *[your answer here]*

**Question 14. Interpreting magnitudes.** Suppose there are roughly 4 million 21-year-olds in the US in a given year. Using your all-cause estimate and the baseline rate, calculate approximately how many **additional deaths per year** this discontinuity implies nationally. Does that seem large or small, and what would you compare it to in order to judge?

In [ ]:
# Your calculation here (the RD estimate is deaths per 100,000 person-years)


**Answer:** *[your answer here]*

**Question 15. Policy discussion.** Some countries, and many US states before the National Minimum Drinking Age Act of 1984, set the drinking age at 18. **Based on your results, what would you predict happens to mortality among 18–20 year olds if the drinking age were lowered to 18?** What considerations beyond this mortality effect should a policymaker weigh (civil liberties, other age-21 policies, enforcement costs, substitution to other risky behaviour)?

**Answer:** *[your answer here]*

## Suggested additional questions (optional)

1. **Log specification.** Re-estimate with `ln(all)` as the outcome. Does the percentage effect change much? When is a log specification preferable for a strictly positive outcome like a mortality rate?
2. **Weighting.** We treated all 50 bins as equally informative. If you knew the person-years behind each bin, how would you incorporate that, and would you expect it to change the point estimate, the standard errors, or both?
3. **External validity across time.** This data covers the late 1990s/early 2000s. Drug overdose deaths among young adults have changed dramatically since. Would the age-21 discontinuity look the same today? Which cause-of-death category would most likely differ, and why?
4. **Comparing designs.** We have now studied a randomized trial (Oregon Medicaid, PS5), propensity-score methods (LaLonde, PS6), a staggered difference-in-differences (social media and mental health, PS7), and a regression discontinuity (drinking age, today). For each: what feature of the data or setting made the design possible, and what is the most important assumption it relies on that the others do not?

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS08"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")